# Preprocessing task 

## Nettoyage des tables 

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")
INTERIM.mkdir(parents=True, exist_ok=True)
NA = ["NA", "NULL", "None", "Not Applicable"]


### Bike types 
il faut :
- garder les 30 types vides avec unknown.
- supprimer les colones max_range (vide a 88%) et battery_capacity (vide a 82%), vehicule_image et description sont inutiles
- Ajouter is_electric 

In [3]:
bike_types = pd.read_csv(RAW / "bike_types.csv", na_values=NA)

bike_types = bike_types.drop(columns=["vehicle_image", "description", "max_range", "battery_capacity"])
bike_types[["form_factor", "propulsion_type"]] = bike_types[["form_factor", "propulsion_type"]].fillna("unknown")
bike_types["is_electric"] = bike_types["propulsion_type"].map(
    {"electric_assist": True, "human": False}).astype("boolean")   # unknown -> <NA>

bike_types.to_parquet(INTERIM / "bike_types.parquet", index=False)


### Bikes :
on a computer_is == 0 pour 13 136, il faut remplacer par NaN et ajouter has_computer

In [4]:
bikes = pd.read_csv(RAW / "bikes.csv", na_values=NA)

bikes["computer_id"] = bikes["computer_id"].replace(0, pd.NA).astype("Int64")
bikes["has_computer"] = bikes["computer_id"].notna()

bikes.to_parquet(INTERIM / "bikes.parquet", index=False)


### Cities : 
- 82 villes sans aucune donnée => il faut les garder dans la table avec un indicateur has_trips

- fuseaux incohérents : 12 villes ES en Europe/Berlin, DE en Europe/Rome, IT, MT, PL et SE en Europe/Berlin => corriger le fuseau à partir du pays (ES → Europe/Madrid, etc.), en gardant Atlantic/Canary pour les Canaries.

In [5]:
cities = pd.read_csv(RAW / "cities.csv", na_values=NA)

country_tz = {"AT": "Europe/Vienna", "CH": "Europe/Zurich", "CY": "Asia/Nicosia",
              "CZ": "Europe/Prague", "DE": "Europe/Berlin", "ES": "Europe/Madrid",
              "GB": "Europe/London", "HR": "Europe/Zagreb", "IT": "Europe/Rome",
              "LV": "Europe/Riga", "MT": "Europe/Malta", "PL": "Europe/Warsaw",
              "SE": "Europe/Stockholm", "SI": "Europe/Ljubljana", "SK": "Europe/Bratislava"}
cities["timezone_raw"] = cities["timezone"]
canary = cities["timezone"] == "Atlantic/Canary"
cities.loc[~canary, "timezone"] = cities.loc[~canary, "country"].map(country_tz)

trip_cities = pd.read_csv(RAW / "trips_small.csv", usecols=["city_id"])["city_id"].unique()
cities["has_trips"] = cities["id"].isin(trip_cities)

cities.to_parquet(INTERIM / "cities.parquet", index=False)


### Station :
- Supprimer la colonne rack_locks : elle contient uniquement des valeurs manquantes (100 %).

- Traiter les valeurs négatives de special_racks : remplacer les valeurs inférieures à 0 (32 stations) par NaN, car elles sont considérées comme invalides.

- Identifier les stations virtuelles : ajouter une colonne booléenne is_virtual pour identifier les stations ayant bike_racks == 0 (1 219 stations).

- Gérer les valeurs manquantes de terminal_type : remplacer les valeurs manquantes (587 stations) par unknown.

- Nettoyer les coordonnées géographiques :
- Supprimer les coordonnées géographiques impossibles ou invalides.
- Conserver les 42 stations situées à plus de 50 km du centre de leur ville, car elles peuvent correspondre à des systèmes régionaux légitimes (ex. : Wolgast, Prora).
- Vérifier les modifications : contrôler les valeurs manquantes, les valeurs aberrantes et la cohérence des données après nettoyage.

In [6]:
stations = pd.read_csv(RAW / "stations.csv", na_values=NA)

stations = stations.drop(columns="rack_locks")
stations.loc[stations["special_racks"] < 0, "special_racks"] = np.nan
stations["terminal_type"] = stations["terminal_type"].fillna("unknown")
stations["is_virtual"] = stations["bike_racks"] == 0
stations["capacity"] = stations["bike_racks"] + stations["special_racks"].fillna(0)

# coordonnées impossibles (hors Europe ou Canaries)
bad_coords = ~stations["lat"].between(27, 72) | ~stations["lon"].between(-25, 45)
print("stations aux coordonnées impossibles :", bad_coords.sum())
stations = stations[~bad_coords]

stations.to_parquet(INTERIM / "stations.parquet", index=False)


stations aux coordonnées impossibles : 1
